<a href="https://colab.research.google.com/github/barrevivo299-design/SMS-Spam-Classification-NaiveBayes/blob/main/SMS_Spam_Classification_NaiveBayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Assignment – Text Classification (NLP)
## Multinomial Naive Bayes – Implementation from Scratch

| Field | Details |
| :--- | :--- |
| **Student** | Bar Revivo \| Last 4 digits of ID: 0076 |
| **Assignment Type** | Text Analysis (NLP) |
| **Learning Type** | Binary Classification (`spam` vs. `ham`) |
| **Algorithm Implemented** | Multinomial Naive Bayes (Implemented from Scratch) |
| **Evaluation Metric** | F1-Score on the `spam` class |
| **Dataset** | [SMS Spam Collection Dataset on Kaggle](https://www.kaggle.com/datasets/hamnawaseem112222222/sms-spam-collection-5572-labeled-sms-messages) |

---

## 1. Prompts, Tools & Additional Sources

Below are the key prompts and resources used throughout the assignment.

### Prompts Used
1. **Prompt:** "Explain how to structure a machine learning pipeline for text classification using Naive Bayes from scratch."
   - **Purpose:** Understanding the workflow and steps required for NLP data preparation.
2. **Prompt:** "How to convert raw text messages into a word frequency matrix without data leakage?"
   - **Purpose:** Implementing Feature Engineering correctly on Train and Test sets.

### Additional Sources
* [scikit-learn Naive Bayes Guide](https://scikit-learn.org/stable/modules/naive_bayes.html)
* [scikit-learn CountVectorizer Reference](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
* [Kaggle SMS Dataset](https://www.kaggle.com/datasets/hamnawaseem112222222/sms-spam-collection-5572-labeled-sms-messages)

---

## 2. Problem Overview & Dataset Description

### Problem Statement
The objective is to build a binary classifier to predict whether an incoming text message is **Spam** (1) or **Ham** (0).

### Dataset Details
* **Source:** SMS Spam Collection Dataset containing 5,572 labeled messages.
* **Class Imbalance:** Mostly legitimate messages (`ham` about 86.6%) with a minority of `spam` (about 13.4%).
* **Evaluation:** Due to imbalance, the model is evaluated using the **F1-Score on the `spam` class**.

In [4]:
import pandas as pd
import numpy as np

# 1. Load Dataset directly from source
url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['Category', 'Message'])

# Display Dataset shape & Class distribution
print(f"גודל ה-dataset: {df.shape}\n")
print("התפלגות המחלקות:")
print(df['Category'].value_counts())
print()

# Display the first 5 rows as a rich Pandas Table
df.head()

גודל ה-dataset: (5572, 2)

התפלגות המחלקות:
Category
ham     4825
spam     747
Name: count, dtype: int64



,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Data Preparation: Deduplication and Single Split

The dataset is provided as a single file and is not pre-split into training and test sets. Therefore, we perform a **single and fixed split** so that all generated rows apply consistently to both subsets throughout the project. No further splits will be performed.

### Key Considerations
* **Data Leakage Prevention:** Duplicate rows are removed prior to splitting. This order is critical: splitting before deduplication could allow identical messages to appear in both the training and testing sets, causing data leakage that artificially inflates performance metrics.
* **Stratified Split:** We use a Stratified split to maintain the class distribution balance across both subsets, along with a fixed `random_state` for full reproducibility.

In [7]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Removed {before - len(df)} duplicate rows. {len(df)} rows remain.')

df['label'] = (df['Category'] == 'spam').astype(int)

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=RANDOM_STATE
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'\ntrain: {train_df.shape}   test: {test_df.shape}')
print(f'spam ratio in train: {100*train_df.label.mean():.2f}%')
print(f'spam ratio in test : {100*test_df.label.mean():.2f}%')

print('\nFirst 5 rows of the trainset:')
display(train_df[['Category', 'Message', 'label']].head())

print('First 5 rows of the test-set:')
display(test_df[['Category', 'Message', 'label']].head())

Removed 403 duplicate rows. 5169 rows remain.

train: (4135, 4)   test: (1034, 4)
spam ratio in train: 12.62%
spam ratio in test : 12.67%

First 5 rows of the trainset:


,Category,Message,label
0,ham,"Ta-Daaaaa! I am home babe, are you still up ?",0
1,ham,Yup having my lunch buffet now.. U eat already?,0
2,ham,Ok anyway no need to change with what you said,0
3,ham,All e best 4 ur exam later.,0
4,ham,Now only i reached home. . . I am very tired n...,0


First 5 rows of the test-set:


,Category,Message,label
0,ham,"Good morning, my Love ... I go to sleep now an...",0
1,ham,"And how you will do that, princess? :)",0
2,ham,"Cool, I'll text you when I'm on the way",0
3,ham,Your right! I'll make the appointment right now.,0
4,ham,Aiya we discuss later lar... Pick ü up at 4 is...,0


## Evaluation Metric

For binary classification problems with a single focus class—such as identifying spam messages (`spam` / `label=1`)—we evaluate model performance using the **F1-Score calculated strictly for the primary class (`spam`)**.

### Metric Justification
* **Class Imbalance:** The dataset is imbalanced (~86% ham vs. ~14% spam). A naive baseline classifier predicting all messages as `ham` would achieve ~86% Accuracy while failing to catch a single spam message. Accuracy is therefore misleading.
* **Balanced Trade-off:** F1-Score represents the harmonic mean of Precision and Recall. It ensures that our model minimizes both false positives (legitimate messages flagged as spam) and false negatives (spam messages missed by the filter).

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, f1_score
import numpy as np

vectorizer = CountVectorizer(stop_words='english', lowercase=True)
X_train = vectorizer.fit_transform(train_df['Message']).toarray()
X_test = vectorizer.transform(test_df['Message']).toarray()

y_train = train_df['label'].values
y_test = test_df['label'].values

class MultinomialNaiveBayesFromScratch:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        self.priors = np.bincount(y) / float(n_samples)
        self.feature_counts = np.zeros((n_classes, n_features))
        for c in self.classes:
            self.feature_counts[c] = X[y == c].sum(axis=0)

        self.class_word_counts = self.feature_counts.sum(axis=1)

    def predict(self, X):
        log_probs = []
        for c in self.classes:
            smoothed_word_probs = (self.feature_counts[c] + self.alpha) / (self.class_word_counts[c] + self.alpha * X.shape[1])
            log_likelihood = X @ np.log(smoothed_word_probs)
            log_prior = np.log(self.priors[c])
            log_probs.append(log_prior + log_likelihood)

        log_probs = np.array(log_probs).T
        return np.argmax(log_probs, axis=1)

model = MultinomialNaiveBayesFromScratch(alpha=1.0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
spam_f1 = f1_score(y_test, y_pred, pos_label=1)

print(f"F1-Score on Spam Class (pos_label=1): {spam_f1:.4f}\n")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

F1-Score on Spam Class (pos_label=1): 0.9333

              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       903
        Spam       0.96      0.91      0.93       131

    accuracy                           0.98      1034
   macro avg       0.97      0.95      0.96      1034
weighted avg       0.98      0.98      0.98      1034



## Feature Engineering

The raw text is transformed into a numerical representation using a five-step processing pipeline:

1. **Text Cleaning & Normalization:** Converting text to lowercase and replacing specific entities with generic tokens (URLs, phone numbers, currency symbols).  
   > **Rationale:** A specific phone number appears only once in the corpus and holds no predictive value, but the presence of *any* phone number is a strong indicator of spam.
2. **Tokenization:** Splitting the text into individual words/tokens.
3. **Stopwords Removal:** Eliminating common words like `at`, `is`, or `the` that appear with similar frequencies across both classes and carry no discriminative information.
4. **Stemming (Porter Stemmer):** Mapping word inflections to a shared root form (`win`, `winning`, `winner` $\rightarrow$ `win`). This reduces vocabulary size and reinforces word frequencies.
5. **Bag of Words:** Constructing a vector representation based on word occurrence counts.

### Data Leakage Prevention
The vocabulary is built using `fit_transform` exclusively on the training set, while only `transform` is applied to the test set. Any word appearing solely in the test set is excluded from the vocabulary—simulating a real-world scenario where the deployed model encounters unseen messages after training.

In [9]:
import re
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer

STOPWORDS = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(t):
    t = t.lower()
    t = re.sub(r'http\S+|www\.\S+', ' urltoken ', t)
    t = re.sub(r'[\w\.-]+@[\w\.-]+\b', ' emailtoken ', t)
    t = re.sub(r'[£$€]', ' currencytoken ', t)
    t = re.sub(r'\b\d{5,}\b', ' phonetoken ', t)
    t = re.sub(r'\d+', ' numtoken ', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def tokenize(t):
    return t.split()

def remove_stopwords(toks):
    return [w for w in toks if w not in STOPWORDS and len(w) > 1]

def stem(toks):
    return [stemmer.stem(w) for w in toks]

def preprocess(t):
    return ' '.join(stem(remove_stopwords(tokenize(clean_text(t)))))

train_df['clean'] = train_df['Message'].apply(preprocess)
test_df['clean'] = test_df['Message'].apply(preprocess)

In [10]:
def show_pipeline(d, name, n=3):
    print('=' * 60)
    print(f'PIPELINE DEMO - {name}')
    print('=' * 60)
    idx = d.index[d['label'] == 1][:1].tolist() + d.index[d['label'] == 0][:2].tolist()
    for i in idx[:n]:
        raw = d.loc[i, 'Message']
        print(f'\n[label = {"spam" if d.loc[i, "label"]==1 else "ham"}]')
        print('1. raw          :', raw[:100])
        print('2. cleaned      :', clean_text(raw)[:100])
        print('3. tokens       :', tokenize(clean_text(raw))[:10])
        print('4. no stopwords :', remove_stopwords(tokenize(clean_text(raw)))[:10])
        print('5. stemmed      :', stem(remove_stopwords(tokenize(clean_text(raw))))[:10])

show_pipeline(train_df, 'TRAIN')
show_pipeline(test_df, 'TEST')

PIPELINE DEMO - TRAIN

[label = spam]
1. raw          : 18 days to Euro2004 kickoff! U will be kept informed of all the latest news and results daily. Unsub
2. cleaned      : numtoken days to euro numtoken kickoff u will be kept informed of all the latest news and results da
3. tokens       : ['numtoken', 'days', 'to', 'euro', 'numtoken', 'kickoff', 'u', 'will', 'be', 'kept']
4. no stopwords : ['numtoken', 'days', 'euro', 'numtoken', 'kickoff', 'kept', 'informed', 'latest', 'news', 'results']
5. stemmed      : ['numtoken', 'day', 'euro', 'numtoken', 'kickoff', 'kept', 'inform', 'latest', 'news', 'result']

[label = ham]
1. raw          : Ta-Daaaaa! I am home babe, are you still up ?
2. cleaned      : ta daaaaa i am home babe are you still up
3. tokens       : ['ta', 'daaaaa', 'i', 'am', 'home', 'babe', 'are', 'you', 'still', 'up']
4. no stopwords : ['ta', 'daaaaa', 'home', 'babe', 'still']
5. stemmed      : ['ta', 'daaaaa', 'home', 'babe', 'still']

[label = ham]
1. raw          : Yup 

In [11]:
vectorizer = CountVectorizer(min_df=2, ngram_range=(1, 1))

X_train = vectorizer.fit_transform(train_df['clean'])
X_test = vectorizer.transform(test_df['clean'])

y_train = train_df['label'].values
y_test = test_df['label'].values

print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')
print(f'X_train shape: {X_train.shape} | X_test shape: {X_test.shape}')
print(f'Sparsity Density: {100 * X_train.nnz / (X_train.shape[0] * X_train.shape[1]):.4f}%')

Vocabulary size: 2573
X_train shape: (4135, 2573) | X_test shape: (1034, 2573)
Sparsity Density: 0.2955%


## Multinomial & Bernoulli Naive Bayes Implementation

### Bayes' Theorem
For a message $d$ and class $c \in \{\text{ham}, \text{spam}\}$:
$$P(c \mid d) \propto P(c) \cdot P(d \mid c)$$

### Naive Assumption
Assuming features are conditionally independent given the class:
$$P(d \mid c) = \prod_{i=1}^{n} P(w_i \mid c)^{\text{count}(w_i, d)}$$

### Laplace Smoothing
To prevent zero probabilities for unseen vocabulary during inference:
$$P(w \mid c) = \frac{\text{count}(w, c) + \alpha}{\sum_{w'} \text{count}(w', c) + \alpha \cdot |V|}$$

### Log-Space Transformation
To prevent numerical underflow caused by multiplying small probabilities, calculations are converted to sums of logarithms:
$$\log P(c \mid d) \propto \log P(c) + \sum_{i=1}^{n} \text{count}(w_i, d) \cdot \log P(w_i \mid c)$$

| Variant | Input Data Representation | Typical Use Case |
| :--- | :--- | :--- |
| **Multinomial** | Word frequencies/counts | Text classification with token frequencies |
| **Bernoulli** | Binary word presence/absence (0 or 1) | Short texts, presence of specific keywords |

In [12]:
import numpy as np
from scipy import sparse

class CustomNaiveBayes:
    """
    Custom implementation of Naive Bayes classifier from scratch.
    Supports both Multinomial and Bernoulli variants with Laplace smoothing.
    """
    def __init__(self, alpha=1.0, variant='multinomial', fit_prior=True):
        if variant not in ('multinomial', 'bernoulli'):
            raise ValueError("variant must be either 'multinomial' or 'bernoulli'")

        self.alpha = float(alpha)
        self.variant = variant
        self.fit_prior = fit_prior

    def fit(self, X, y):
        X = sparse.csr_matrix(X, dtype=np.float64)

        if self.variant == 'bernoulli':
            X = (X > 0).astype(np.float64)

        self.classes_ = np.unique(y)
        n_samples, n_features = X.shape
        n_classes = len(self.classes_)

        self.class_log_prior_ = np.zeros(n_classes)
        self.feature_log_prob_ = np.zeros((n_classes, n_features))

        if self.variant == 'bernoulli':
            self.feature_log_prob_neg_ = np.zeros((n_classes, n_features))

        for i, c in enumerate(self.classes_):
            X_c = X[y == c]
            n_c = X_c.shape[0]

            # Compute Prior Log Probabilities
            if self.fit_prior:
                self.class_log_prior_[i] = np.log(n_c / float(n_samples))
            else:
                self.class_log_prior_[i] = np.log(1.0 / n_classes)

            # Compute Feature Likelihoods with Laplace Smoothing
            feature_counts = np.asarray(X_c.sum(axis=0)).ravel()

            if self.variant == 'multinomial':
                smoothed_numerator = feature_counts + self.alpha
                smoothed_denominator = feature_counts.sum() + (self.alpha * n_features)
                self.feature_log_prob_[i] = np.log(smoothed_numerator / smoothed_denominator)
            else:  # Bernoulli
                prob = (feature_counts + self.alpha) / (n_c + 2.0 * self.alpha)
                self.feature_log_prob_[i] = np.log(prob)
                self.feature_log_prob_neg_[i] = np.log(1.0 - prob)

        return self

    def _joint_log_likelihood(self, X):
        X = sparse.csr_matrix(X, dtype=np.float64)

        if self.variant == 'bernoulli':
            X = (X > 0).astype(np.float64)
            # Log likelihood incorporating both present and absent features
            joint_ll = X @ (self.feature_log_prob_ - self.feature_log_prob_neg_).T
            joint_ll += self.feature_log_prob_neg_.sum(axis=1)
        else:
            joint_ll = X @ self.feature_log_prob_.T

        return joint_ll + self.class_log_prior_

    def predict(self, X):
        jll = self._joint_log_likelihood(X)
        return self.classes_[np.argmax(jll, axis=1)]

    def predict_proba(self, X):
        jll = self._joint_log_likelihood(X)
        # Numerical stability using Log-Sum-Exp trick
        max_log = np.max(jll, axis=1, keepdims=True)
        exp_jll = np.exp(jll - max_log)
        return exp_jll / exp_jll.sum(axis=1, keepdims=True)